In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from itertools import product

# MACD Calculation
def macd(data, fast, slow, signal):
    data['EMA_Fast'] = data['close'].ewm(span=fast, min_periods=1, adjust=False).mean()
    data['EMA_Slow'] = data['close'].ewm(span=slow, min_periods=1, adjust=False).mean()
    data['MACD'] = data['EMA_Fast'] - data['EMA_Slow']
    data['Signal_Line'] = data['MACD'].ewm(span=signal, min_periods=1, adjust=False).mean()
    return data

# Backtesting Function
def backtest(data):
    buy_signals = (data['MACD'] > data['Signal_Line']) & (data['MACD'].shift(1) <= data['Signal_Line'].shift(1))
    sell_signals = (data['MACD'] < data['Signal_Line']) & (data['MACD'].shift(1) >= data['Signal_Line'].shift(1))
    
    positions = pd.Series(index=data.index, data=np.nan)
    positions[buy_signals] = 1
    positions[sell_signals] = 0
    positions.ffill(inplace=True)
    positions.fillna(0, inplace=True)
    
    data['Strategy'] = positions.shift(1) * (data['close'].pct_change())
    return data['Strategy'].cumsum().iloc[-1]

# Parameter Optimization
def optimize_macd(data, fast_range, slow_range, signal_range):
    best_params = None
    best_profit = -np.inf
    
    for fast, slow, signal in product(fast_range, slow_range, signal_range):
        if fast >= slow:
            continue
        temp_data = macd(data.copy(), fast, slow, signal)
        profit = backtest(temp_data)
        
        if profit > best_profit:
            best_profit = profit
            best_params = (fast, slow, signal)
    
    return best_params, best_profit

if __name__ == "__main__":
    # Download historical data for a stock
    ticker = "AAPL"
    data = yf.download(ticker, start="2020-01-01", end="2023-01-01")
    
    # Define parameter ranges
    fast_range = range(2, 50, 1)
    slow_range = range(2, 50, 1)
    signal_range = range(2, 21, 1)
    
    # Optimize MACD parameters
    best_params, best_profit = optimize_macd(data, fast_range, slow_range, signal_range)
    
    print(f"Best Parameters: Fast EMA={best_params[0]}, Slow EMA={best_params[1]}, Signal Line={best_params[2]}")
    print(f"Net Profit: {best_profit}")
